# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## 제출 정보
- 이름: 김지인
- GitHub ID: jiin-0
- 작성일: 2026-11-11
- 최종 Notebook URL: https://github.com/jiin-0/llm-data-analysis-study/blob/main/chapter09/ch09_regression_analysis.ipynb


(영상을 시청한 뒤에도 회귀 분석에 대한 개념이 명확히 잡혀있지 않아 이번 실습에서는 예시 분석 틀을 따라가면서 단계와 개념들을 이해해보겠습니다)
## 1. 예측 문제 정의
- Target: order_total
- 예측 단위: 주문 1건
- 예측 시점: 주문 개시(결제 전)
- 예측 시점에 알 수 있는 정보: 주문 메타데이터와 고객의 비식별 특성은 알고 있음.
- 예측 시점에 알 수 없는 정보: 주문 상세 수량, 단가, 금액은 모델 입력에 제공하지 않음.

### 나의 해석과 판단
추후 이어질 Data leakage를 차단하기에 충분한 문제 정의이다. 또한 order_total은 연속형 수치 데이터이브로 회귀분석 성격에 부합한다.

---

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [3]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: C:\dev\llm-data-analysis-study
PROCESSED_DIR: C:\dev\llm-data-analysis-study\data\processed


## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [4]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


## 2. 검증된 전처리 데이터를 불러옵니다

In [5]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [6]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-10-14,bank_transfer,completed,10,1,361000,M,67,대구
1,79,62,2025-10-16,bank_transfer,refunded,10,3,283000,F,63,광주
2,12,98,2025-10-17,kakao_pay,completed,10,4,444000,F,61,인천
3,169,9,2025-10-17,kakao_pay,completed,10,4,448000,M,69,서울
4,152,62,2025-10-18,naver_pay,refunded,10,5,985000,F,63,광주


Model rows: 300


## 4. Feature Audit을 확인합니다

In [7]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 2. Feature Leakage Audit
| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | O | 교육용 예측 시점에 사용 가능하다고 가정 |
| order_dayofweek | O | X | X | O | 교육용 예측 시점에 사용 가능하다고 가정 |
| age | O | X | X | O | 교육용 예측 시점에 사용 가능하다고 가정 |
| payment_method | O | X | X | O | 교육용 예측 시점에 사용 가능하다고 가정 |
| gender | O | X | X | O | 교육용 예측 시점에 사용 가능하다고 가정 |
| city | O | X | X | O | 교육용 예측 시점에 사용 가능하다고 가정 |
| avg_unit_price | X | O | O | X | 주문 상세에서 만든 목표 대리 변수 |
| customer_id | O | X | O | X | 고객 식별자 |
| item_count | X | X | O | X | 주문 상세가 확인된 뒤 계산되는 사후 집계 |
| line_total | X | O | O | X | 목표값을 구성하는 주문 상세 금액 |
| order_id | O | X | O | X | 주문 식별자 |
| order_status | X | X | O | X | 예측 시점 이후에 확정될 수 있는 사후 정보 |
| order_total | X | O | O | X | 예측 대상 자체 |
| product_id | X | X | O | X | 주문 상세가 확인되어야 알 수 있는 식별자 |
| quantity | X | O | O | X | 목표값 계산 재료 |
| total_quantity | X | O | O | X | 주문 상세에서 만든 목표 대리 변수 |
| unit_price | X | O | O | X | 목표값 계산 재료 |

- 금지 feature overlap 개수: 0개
- 가장 위험하다고 판단한 feature: line_total
- 이유: target인 order_total을 직접 구성하는 주문 상세 금액으로, 모델이 패턴 학습 대신 정답을 그대로 유추하게 만드는 값이 된다.

![Leakage 점검](images/step02_leakage.png)

---

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity × unit_price
order_total = order_id별 line_total 합계
```

- line_total 검증 결과: line_total = quantity × unit_price로 적절히 계산되었다.(오류 발생X)
- orders ↔ target 관계 검증 결과: one_to_one
- orders → customers 관계 검증 결과: many_to_one

### 나의 해석과 판단
관계 검증 없이 병합 수행 시 행이 줄거나 늘었을 때 오류없이 진행된 것인지 파악하기 어려워진다. 따라서 병합 시 관계를 확인하여 기대 병합 dataframe의 사이즈를 알고 있어야한다. 즉, 관계 검증을 하지 않을 경우 결과에 대한 평가를 진행할 수 없어지므로 필수적이다. 또한, 이 과정을 통해 데이터에대한 이해를 높일 수 있고, 추후 발생할 문제나 분석에도 유연하게 대처할 수 있다.

---

## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [8]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-10-14,2026-07-19
1,test,56,18.67,2026-07-20,2026-10-13


Chronological split gate: PASS


In [9]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

Train: (244, 6) Final Test: (56, 6)


## 4. Train / Final Test 분할

- Train 시작일: 2025-10-14
- Train 종료일: 2026-07-19
- Train 행 수: 244
- Final Test 시작일: 2026-07-20
- Final Test 종료일: 2026-10-13
- Final Test 행 수: 56
- 같은 달력 날짜 중복 여부: X

### 분할 판단
이번 분석에서는 시간 순서 분할을 진행하였다. 시간 순서 분할을 사용하는 이유는 과거 주문 이력을 학습해 미래의 주문 금액을 예측하는 환경을 재현하고 시간적 데이터 누수를 막기 위함이다. 주문 데이터는 시간의 흐름에 다라 순차적으로 발생한다. 따라서 무작위 분할을 적용해 미래 시점의 주문 내역이나 동일 고객의 이후 구매 패턴이 학습에 섞이며, 모델의 일반화 성능이 왜곡될 수 있다. 이를 방지하기 위해 데이터를 시간순으로 정렬한 뒤, 모델 생성용 train과 추후 성능 검증용 final test 시간으로 나누어 구성하였다.

![시간 분할](images/step04_split.png)

---

## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [10]:
models = make_regression_models(random_state=42)
print(list(models))

['Baseline Mean', 'Linear Regression', 'Random Forest']


## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [11]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,432319.498508,27358.549792,-0.098256
2,Linear Regression,5,630345.690174,402292.827278,-1.910478


## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 423004.808743 | 44153.043466 | -0.022821 | |
| Random Forest | 432319.498508 | 27358.549792 | -0.098256 | |
| Linear Regression | 630345.690174 | 402292.827278 | -1.910478 | |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest
- 선택 기준: 두 비베이스라인 모델 모두 CV MAE 평균은 Baseline Mean(423,005)을 넘지 못했으나, Random Forest(432319)가 Linear Regression(630345)보다 평균 오차와 CV R²에서 크게 우수하며, CV MAE 표준편차(27358) 또한 Baseline Mean(44153)보다 작아 시점 변화에 따른 예측 변동성이 가장 낮고 안정적이므로 선정하였다.
- Final Test 결과를 보기 전에 선택했는가: 예 

### 나의 해석과 판단
CV 평균은 모델이 평균적으로 어느 정도의 오차를 내는지 보여주는 지표로, 작을수록 예측일 정확하다는 것을 의미한다.
CV 표준편차는 모델 성능이 시점이 달라질 때 얼마나 일정하게 유지되는지를 보여주는 안정성 지표로 작을수록 일관된 성능을 낸다는 것을 의미한다.

따라서 평균만 관찰하게 되면 모델의 안정성을 확인할 수 없기 때문에 표준편차를 같이 확인하여 목적에 맞는 모델을 선정해야한다.

---

## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [12]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [13]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,324458.846890,467568.128073,549046.076913,-0.106745,-5.44


## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416266.191884 | 443449.648712 | 522189.699620 | -0.001121 | 0.00 |
| Frozen Selected Model | selected_by_train_cv |324458.846890 | 467568.128073 | 549046.076913 | -0.106745 | -5.44 |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
baseline mean 대비 train에서 MAE는 감소하였고, TEST에서는 MAE가 증가하였다. test에서 R2값 또한, random forest에서 감소하였으며 최종 MAE improvement는 음수값으로 나타났다.

### 나의 해석과 판단
frozen model은 baseline대비 TEST에서 MAE, RMSE가 증가하였다. 또한, R2는 감소하였다. 따라서 baseline 대비 개선되지 않고 더 낮은 일반화 성능을 보였다. 이는 앞선 train 단계에서 학습한 결과가 미래 시점의 테스트 데이터 예측 결과에 좋은 결과를 내지 못한 것이다.
MAE와 RMSE의 차이는 RMSE는 오차를 제곱한 뒤 평균내는 값으로 큰 오차에 훨씬 민감하게 반응하는 지표이다. 따라서 모든 주문에 대해 오차가 고르다면 두 지표가 비슷하게 나타나지만 RMSE가 MAE보다 뚜렷한 경우 일부 주문에서 평균을 크게 벗어나는 큰 예측 오차가 발생하고 있다는 것을 의미하게 된다. frozen selected model에서 RMSE값이 더 증가하였으므로, 성능이 더 악화되었다는 지표가 된다.
R2는 결정계수로 단순 평균으로 예측했을 때의 오차 대비 모델이 타겟의 변동을 얼마나 설명하는 지를 나타낸다. 0이면 평균 예측과 동일한 수준이고, 음수 값이 나왔다는 것은 단순 평균값 하나로 예측할 때보다 오차가 더 크다는 것이다. 따라서 현재의 6개의 기본 feature만으로는 주문 총액의 변동을 설명하기 어렵다는 것을 의미한다.


### 업무·분석적 의미
선정한 feature를 기반으로 주문 총액을 예상했을 때 baseline mean보다 성능이 떨어지며, R2값 또한 음수로 나타났다. 따라서 현재의 feature로는 주문 총액의 변동을 설명하기 어렵다는 뜻이다. 따라서 예측 성능을 확보하려면 기본 메타데이터 이외의 데이터를 추가하여야 한다.

---

### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [13]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-09-20,1877000,5.126427e+05,1.364357e+06,1.364357e+06,Random Forest
1,96,2026-08-16,1690000,6.773614e+05,1.012639e+06,1.012639e+06,Random Forest
2,21,2026-09-24,1767000,7.566429e+05,1.010357e+06,1.010357e+06,Random Forest
3,145,2026-09-01,1863000,8.898975e+05,9.731025e+05,9.731025e+05,Random Forest
4,183,2026-08-28,1685000,7.664431e+05,9.185569e+05,9.185569e+05,Random Forest
5,15,2026-08-28,1677000,8.113307e+05,8.656693e+05,8.656693e+05,Random Forest
6,194,2026-08-24,257000,1.083625e+06,-8.266253e+05,8.266253e+05,Random Forest
7,168,2026-09-16,10000,8.252255e+05,-8.152255e+05,8.152255e+05,Random Forest
8,282,2026-07-30,1566000,7.544169e+05,8.115831e+05,8.115831e+05,Random Forest
9,179,2026-10-13,86000,8.897504e+05,-8.037504e+05,8.037504e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


## 7. 대표 오차 사례

### 사례 1
- 실제값: 1877000
- 예측값: 512642.7
- residual: 1364357
- absolute error: 1364357
- 관찰: 2026-09-20에 발생한 100만원이 넘는 고액 주문을 오십 만원대로 예측하여 전체 테스트 데이터 중 가장 큰 오차가 발생하였다. 상위 오차 표 전체에서도 100만원이 넘는 고액의 주문을 50~80만원대로 낮게 예측하는 경향이 관찰된다.
- 가능한 이유 후보: 데이터 누수 방지를 위해 unit_price, quantity, item_count를 제외했기 때문에, 동일한 시점에서 인구통계 조건을 가진 일반 주문과 고액 주문을 구분할 정보가 부족했을 가능성이 있다.
- 추가 확인: 다른 고액의 주문들(100만원 이상) 또한, 50-80만원대로 낮게 예측이 되고 있는 것을 보아 현재, 고액의 주문을 제대로 예측하지 못하고 있다고 추측할 수 있다. 또한, 사후 오차 분석 차원에서 csv 테이블을 조회해 단가 상승 요인인지 다량 구매 요인인지 확인할 필요가 있다.

### 사례 2
- 실제값: 1690000
- 예측값: 677361.4
- residual: 1012639
- absolute error: 1012639
- 관찰: 2026-08-16에 발생한 100만원이 넘는 고액 주문을 육십 만원대로 예측하여 두 번째로 큰 과소예측 사례이다. 사례 1에 비해서는 예측값이 약 16.5만원 높게 산출되었으나, 여전히 실제 고액 규모를 절반 수준도 반영하지 못했다.
- 가능한 이유 후보: 과대 예측 사례(2026-09-16의 10,000원)와 함께 볼 때, 모델이 개별 주문의 실제 규모 차이를 구분하지 못하고 유사 조건 그룹의 중간값(50만~90만원대)으로 일괄 수렴하여 예측하고 있을 가능성이 있다.
- 추가 확인: 오차 상위 행들이 특정 월(`8~9월`)이나 특정 고객군에 집중되어 있는지 분포 점검할 필요가 있다.

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

---

## 11. 자동 Validation Evidence를 만듭니다

In [14]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목
- 없음 / 있음: 없음
- 있다면 원인과 수정 내용: X

---

## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [15]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> C:\dev\llm-data-analysis-study\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\dev\llm-data-analysis-study\reports\ch09_regression_split_summary.csv
feature_audit -> C:\dev\llm-data-analysis-study\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\dev\llm-data-analysis-study\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\dev\llm-data-analysis-study\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\dev\llm-data-analysis-study\reports\ch09_regression_predictions_internal.csv
validation -> C:\dev\llm-data-analysis-study\reports\ch09_regression_validation.csv
checklist -> C:\dev\llm-data-analysis-study\reports\ch09_regression_checklist.csv
report -> C:\dev\llm-data-analysis-study\reports\ch09_regression_report.md
actual_vs_predicted_figure -> C:\dev\llm-data-analysis-study\reports\figures\ch09_actual_vs_predicted.png
residual_figure -> C:\dev\llm-data-analysis-study\reports\figures\ch09_residual_histogr

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [O] `ch09_regression_split_summary.csv`
- [O] `ch09_regression_feature_audit.csv`
- [O] `ch09_regression_cv_summary.csv`
- [O] `ch09_regression_model_comparison.csv`
- [O] `ch09_regression_validation.csv`
- [O] `ch09_regression_checklist.csv`

### Internal 파일
- [O] `ch09_regression_model_data_internal.csv`
- [O] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [O] `ch09_regression_report.md`
- [O] `ch09_actual_vs_predicted.png`
- [O] `ch09_residual_histogram.png`

### 공개 범위 판단
order_id는 실제 데이터를 특정하는 key값이 되므로 외부에 공개하거나 model학습에 사용해서는 안되므로 internal로 용도를 한정해야한다.

---

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [O] 현재 데이터로는 사용 보류

### 판단 근거
1. R2값이 음수로 나타났으며 Baseline mean 대비 MAE 값과 RMSE값이 크게 나타났다.
2. 극단적인 주문들에 대해 예측 능력이 다소 떨어짐
3. 위의 내용을 토대로 현재의 feature로는 order_total이 제대로 예측되지 않고 있다고 판단됨.

### 업무적 위험
예측 정확도가 너무 낮으므로, 예측 결과를 신뢰할 수 없어 업무적 활용도 자체가 낮다. 따라서 예측 결과를 활용할 경우 판단의 신뢰도가 떨어진다.

### 현재 데이터의 한계
주어진 feature만으로는 order_total을 예측하기에 한계가 있다. 따라서 다른 정보가 추가적으로 필요하다.

### 다음 개선 우선순위
1. feature 추가
2. model 규제하여 극단값에 분기가 과도하게 쪼개지는 현상을 방지(이하 제미나이에게 도움을 구했습니다.)
3. 타겟 변수를 로그 변환하여 스케일 편차를 줄여 예측 변별력을 개선한다.

---

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [16]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: O
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: PASS
- Notebook 결과와 스크립트 결과의 일관성: O

---

## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 15. 완료 체크

- [O] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [O] Target과 예측 시점을 설명할 수 있다.
- [O] 금지 feature가 입력에 없음을 확인했다.
- [O] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [O] 전처리는 Pipeline 내부에서 학습된다.
- [O] Train TimeSeriesSplit으로 후보를 비교했다.
- [O] Final Test 전에 Selected Model을 고정했다.
- [O] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [O] MAE, RMSE, R²를 함께 읽었다.
- [O] 내부 식별자 결과와 공개 보고서를 구분했다.
- [O] Validation Evidence가 모두 PASS다.

## 최종 체크

- [O] Target과 예측 시점을 정의했습니다.
- [O] Leakage Audit을 수행했습니다.
- [O] Target 계산과 관계 검증을 확인했습니다.
- [O] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [O] 전처리가 Pipeline 내부에서 학습됩니다.
- [O] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [O] Final Test 전에 Selected Model을 고정했습니다.
- [O] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [O] MAE, RMSE, R²를 함께 해석했습니다.
- [O] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [O] Validation Evidence가 모두 PASS입니다.
- [O] Internal 결과와 공개 결과를 구분했습니다.
- [O] 낮은 성능도 숨기지 않았습니다.
- [O] 전체 스크립트를 재실행했습니다.
- [O] 최종 Notebook URL을 제출합니다.